In [ ]:
"""
SMS SPAM DETECTION - COMPLETE PREPROCESSING SCRIPT
====================================================
This script loads, cleans, preprocesses, and prepares SMS data for ML training
"""

import pandas as pd
import numpy as np
import re
import nltk
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)


def clean_sms_text(text):
    """
    Clean SMS text - ONLY clean the message, DO NOT replace URL, Email, Phone
    Keep all original content, just make it readable
    """
    if pd.isna(text):
        return ""

    text = str(text)

    # ============ 1. EXPAND SMS ABBREVIATIONS ============
    abbreviations = {
        'u': 'you', 'r': 'are', 'ur': 'your', 'urs': 'yours',
        'd': 'the', 'y': 'why', 'b': 'be', 'c': 'see', 'k': 'okay',
        'n': 'and', 'w': 'with', 'w/o': 'without', 'w/': 'with',
        '4': 'for', '2': 'to', '2day': 'today', '2moro': 'tomorrow',
        '2nite': 'tonight', 'b4': 'before', 'pls': 'please',
        'plz': 'please', 'thx': 'thanks', 'ty': 'thank you',
        'lol': 'laugh out loud', 'brb': 'be right back',
        'btw': 'by the way', 'idk': 'i do not know',
        'imo': 'in my opinion', 'irl': 'in real life',
        'jk': 'just kidding', 'lmk': 'let me know',
        'np': 'no problem', 'omg': 'oh my god',
        'gtg': 'got to go', 'tbh': 'to be honest',
        'dm': 'direct message', 'pm': 'private message',
        'asap': 'as soon as possible', 'afk': 'away from keyboard',
        'bday': 'birthday', 'cya': 'see you', 'fyi': 'for your information',
        'hmu': 'hit me up', 'ily': 'i love you',
        'lmao': 'laughing my ass off', 'rofl': 'rolling on floor laughing',
        'smh': 'shaking my head', 'ttyl': 'talk to you later',
        'wyd': 'what are you doing', 'wtf': 'what the fuck',
        'yk': 'you know', 'yw': 'you\'re welcome',
        'omw': 'on my way', 'da': 'the', 'ya': 'you',
        'l8r': 'later', 'gr8': 'great', 'g2g': 'got to go',
        'bbl': 'be back later', 'bbs': 'be back soon',
        'msg': 'message', 'txt': 'text', 'fav': 'favorite',
        'sec': 'second', 'mins': 'minutes', 'hrs': 'hours',
        'ppl': 'people', 'bc': 'because', 'cuz': 'because',
        'tbh': 'to be honest', 'idc': 'i do not care',
        'ily': 'i love you', 'ilu': 'i love you',
    }

    # Sort by length (longest first) to avoid partial replacements
    for short, full in sorted(abbreviations.items(), key=lambda x: len(x[0]), reverse=True):
        text = re.sub(r'\b' + re.escape(short) + r'\b', full, text, flags=re.IGNORECASE)

    # ============ 2. FIX COMMON ISSUES ============
    # Fix spacing around punctuation
    text = re.sub(r'\s+([!?.])', r'\1', text)
    text = re.sub(r'([!?.])\s+', r'\1 ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove repeated punctuation
    text = re.sub(r'([!?.])\1+', r'\1', text)

    # Remove multiple dots
    text = re.sub(r'\.{2,}', '.', text)

    # ============ 3. CONVERT TO LOWERCASE ============
    text = text.lower()

    return text


def extract_features_from_text(df):
    """
    Extract features from cleaned text messages
    """
    print("\nStep 7: Extracting features from text...")

    # Word count
    df['word_count'] = df['cleaned_message'].str.split().str.len()

    # Character count
    df['char_count'] = df['cleaned_message'].str.len()

    # Uppercase ratio (check original text for uppercase preservation)
    df['upper_ratio'] = df['cleaned_message'].apply(
        lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)
    )

    # Exclamation count - using count method correctly
    df['exclamation_count'] = df['cleaned_message'].apply(lambda x: str(x).count('!'))

    # Question count - using count method correctly
    df['question_count'] = df['cleaned_message'].apply(lambda x: str(x).count('?'))

    # Digit count - using count method with string
    df['digit_count'] = df['cleaned_message'].apply(lambda x: sum(1 for c in str(x) if c.isdigit()))

    # URL presence - using str.contains with proper escaping
    df['has_url'] = df['cleaned_message'].str.contains(r'https?://|www\.', case=False, na=False).astype(int)

    # Email presence
    df['has_email'] = df['cleaned_message'].str.contains(r'@', na=False).astype(int)

    # Phone presence - using proper regex
    df['has_phone'] = df['cleaned_message'].str.contains(r'\d{3}[-.]?\d{3}[-.]?\d{4}', na=False).astype(int)

    print("Feature extraction complete.")
    print(f"Features added: {['word_count', 'char_count', 'upper_ratio', 'exclamation_count', 'question_count', 'digit_count', 'has_url', 'has_email', 'has_phone']}")

    return df


def load_and_clean_sms_dataset(csv_path, output_path):
    """
    Complete preprocessing pipeline for SMS spam detection dataset

    Parameters:
    -----------
    csv_path : str
        Path to the input CSV file
    output_path : str
        Path where the cleaned dataset will be saved

    Returns:
    --------
    df_final : pandas.DataFrame
        The cleaned and preprocessed dataset
    """

    # ── Step 1: Load CSV file ─────────────────────────────
    print("="*60)
    print("SMS SPAM DETECTION - PREPROCESSING PIPELINE")
    print("="*60)
    print("\nStep 1: Loading dataset...")
    df = pd.read_csv(csv_path)

    # Remove unnamed columns
    unnamed_cols = [col for col in df.columns if col.startswith('Unnamed:')]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)
        print(f"Removed unnamed columns: {unnamed_cols}")

    # Also remove URL, EMAIL, PHONE columns if they exist (we keep them in text)
    cols_to_drop = ['URL', 'EMAIL', 'PHONE']
    for col in cols_to_drop:
        if col in df.columns:
            df = df.drop(columns=[col])
            print(f"Removed column: {col}")

    print(f"Original shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Label distribution:\n{df['label'].value_counts()}")

    # ── Step 2: Standardize labels ──────────────────────────
    print("\nStep 2: Standardizing labels...")
    df['label'] = df['label'].str.lower()
    print(f"Label distribution after standardization:\n{df['label'].value_counts()}")

    # ── Step 3: Clean the text messages ────────────────────
    print("\nStep 3: Cleaning text messages...")
    df['cleaned_message'] = df['text'].apply(clean_sms_text)
    print("Text cleaning complete.")

    # Show sample
    print("\nSample of cleaned messages:")
    print(df[['label', 'cleaned_message']].head(5))

    # ── Step 4: Handle missing values ──────────────────────
    print("\nStep 4: Handling missing values...")

    # Drop rows with missing label or text
    before = len(df)
    df = df.dropna(subset=['label', 'text'])
    after = len(df)
    print(f"Dropped {before - after} rows with missing values.")
    print(f"Shape after handling missing: {df.shape}")

    # ── Step 5: Remove empty messages ──────────────────────
    print("\nStep 5: Removing empty messages...")
    before = len(df)
    df = df[df['cleaned_message'].str.len() > 2]
    after = len(df)
    print(f"Removed {before - after} empty messages.")
    print(f"Shape after removing empty messages: {df.shape}")

    # ── Step 6: Remove duplicates ──────────────────────────
    print("\nStep 6: Removing duplicates...")
    before = df.shape[0]
    df = df.drop_duplicates(subset=['cleaned_message'])
    after = df.shape[0]
    print(f"Removed {before - after} duplicate rows.")
    print(f"Shape after removing duplicates: {df.shape}")

    # ── Step 7: Encode labels ──────────────────────────────
    print("\nStep 7: Encoding labels...")
    label_encoder = LabelEncoder()
    df['label_encoded'] = label_encoder.fit_transform(df['label'])
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
    print(f"Label mapping: {label_mapping}")

    # ── Step 8: Extract features from text ──────────────────
    df = extract_features_from_text(df)

    # ── Step 9: Separate features and target ──────────────
    print("\nStep 9: Separating features and target...")

    # Select features for training
    feature_columns = ['word_count', 'char_count', 'upper_ratio',
                       'exclamation_count', 'question_count', 'digit_count',
                       'has_url', 'has_email', 'has_phone']

    # Check if all feature columns exist
    missing_features = [col for col in feature_columns if col not in df.columns]
    if missing_features:
        print(f"Warning: Missing features: {missing_features}")
        feature_columns = [col for col in feature_columns if col in df.columns]

    X = df[feature_columns].copy()
    y = df['label_encoded'].copy()

    print(f"Features shape: {X.shape}")
    print(f"Target shape: {y.shape}")
    print(f"Features: {feature_columns}")

    # ── Step 10: Check class imbalance ───────────────────────
    print("\nStep 10: Checking class imbalance...")
    distribution = y.value_counts()
    print(f"Class distribution:\n{distribution}")

    # Map back to original labels for clarity
    label_names = {v: k for k, v in label_mapping.items()}
    for label, count in distribution.items():
        print(f"  {label_names[label]}: {count} ({count/len(y)*100:.1f}%)")

    ratio = distribution.min() / distribution.max() if distribution.max() > 0 else 0
    print(f"Minority/Majority ratio: {ratio:.2f}")

    if ratio < 0.8:
        print("Class imbalance detected! Applying SMOTE...")
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)
        print(f"Shape after SMOTE: {X_resampled.shape}")
        print(f"Class distribution after SMOTE:\n{pd.Series(y_resampled).value_counts()}")
        X = X_resampled
        y = y_resampled
    else:
        print("Classes are balanced. SMOTE not needed.")
        X_resampled = X
        y_resampled = y

    # ── Step 11: Feature scaling with MinMaxScaler ──────────
    print("\nStep 11: Applying MinMaxScaler...")
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
    print("Scaling complete. All features now in range [0, 1].")
    print(f"Sample scaled values:\n{X_scaled.head(3)}")

    # ── Step 12: Split data for training/testing ────────────
    print("\nStep 12: Splitting data for training and testing...")
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
    )
    print(f"Training set: {X_train.shape}")
    print(f"Test set: {X_test.shape}")

    # ── Step 13: Combine and save ────────────────────────────
    print("\nStep 13: Saving cleaned dataset...")

    # Create final dataframe with clean columns
    df_final = X_scaled.copy()
    df_final['label'] = y_resampled.map({v: k for k, v in label_mapping.items()})
    df_final['label_encoded'] = y_resampled

    # Save the feature dataset
    df_final.to_csv(output_path, index=False)
    print(f"Cleaned dataset saved to: {output_path}")
    print(f"Final shape: {df_final.shape}")
    print(f"Final columns: {df_final.columns.tolist()}")

    # ── Step 14: Summary ─────────────────────────────────────
    print("\n" + "="*60)
    print("PREPROCESSING SUMMARY")
    print("="*60)
    print(f"Total messages: {len(df)}")
    print(f"Spam messages: {len(df[df['label']=='spam'])}")
    print(f"Ham messages: {len(df[df['label']=='ham'])}")
    print(f"Features extracted: {len(feature_columns)}")
    print(f"Training samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")
    print(f"SMOTE applied: {'Yes' if ratio < 0.8 else 'No'}")
    print("="*60)

    return df_final, X_train, X_test, y_train, y_test, scaler


if __name__ == "__main__":
    # Define paths - Using your exact path
    csv_path = "/content/drive/MyDrive/8th sem major project/sms_scam_detection_dataset_merged_with_lang1.csv"
    output_path = "/content/drive/MyDrive/8th sem major project/sms_cleaned_preprocessed.csv"

    print(f"Input file: {csv_path}")
    print(f"Output file: {output_path}")
    print()

    # Run preprocessing
    df_final, X_train, X_test, y_train, y_test, scaler = load_and_clean_sms_dataset(
        csv_path, output_path
    )

    # Print final summary
    print("\n" + "="*60)
    print("✅ PREPROCESSING COMPLETE!")
    print("="*60)
    print(f"Final dataset saved to: {output_path}")
    print(f"Training samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")
    print(f"Features: {df_final.columns.tolist()}")
    print("="*60)

Input file: /content/drive/MyDrive/8th sem major project/sms_scam_detection_dataset_merged_with_lang1.csv
Output file: /content/drive/MyDrive/8th sem major project/sms_cleaned_preprocessed.csv

SMS SPAM DETECTION - PREPROCESSING PIPELINE

Step 1: Loading dataset...
